In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from keras.datasets import fashion_mnist
import wandb

# Load dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## Question 1
Download the fashion-MNIST dataset and plot 1 sample image for each class.

Use from keras.datasets import fashion_mnist for getting the fashion mnist dataset.

In [3]:
# Initialize wandb
wandb.init(project="DA6401-Assignment1", name="Backprop and gradient descent")

# Define a function to plot samples
def plot_samples_for_index(sample_index):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()

    for class_id in range(10):
        # Find indices of samples belonging to this class
        class_indices = np.where(y_train == class_id)[0]

        # Make sure we don't exceed array bounds by using modulo
        safe_index = sample_index % len(class_indices)
        selected_idx = class_indices[safe_index]

        # Display the image
        axes[class_id].imshow(X_train[selected_idx], cmap='gray')
        axes[class_id].set_title(f"{class_names[class_id]}\nSample #{safe_index}")
        axes[class_id].axis('off')

    plt.tight_layout()
    return fig

# Create a custom panel in wandb to control the sample index
# Define a range of indices to explore
sample_indices = list(range(0, 35, 5))  # From 0 to 35, steps of 5

# Log multiple visualizations for different sample indices
for idx in sample_indices:
    fig = plot_samples_for_index(idx)

    # Log to wandb with the specific index as the step
    wandb.log({
        "Class Samples": wandb.Image(fig)
    }, step=idx)

    plt.close(fig)

# print("Finished logging sample visualizations to Wandb.")
wandb.finish()


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krishvamsi321 (krishvamsi321-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Question 2 :
Implement a feedforward neural network which takes images from the fashion-mnist data as input and outputs a probability distribution over the 10 classes.

Your code should be flexible such that it is easy to change the number of hidden layers and the number of neurons in each hidden layer.

In [4]:
class NeuralNetwork:
    def __init__(self, input_size=784, hidden_layers=[128, 64], output_size=10, activation='relu', init_method='xavier'):
        """
        Parameters:
        - input_size: Input dimension (default: 784 for 28x28 images)
        - hidden_layers: List specifying number of neurons in each hidden layer
        - output_size: Number of output classes (default: 10 for Fashion-MNIST)
        - activation: Activation function ('sigmoid', 'tanh', or 'relu' as these are mentioned in document)
        - init_method: Weight initialization method ('random' or 'xavier' as these are mentioned in document)
        """
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.output_size = output_size
        self.activation = activation
        self.init_method = init_method

        # Architecture with input, hidden, and output layers
        layer_sizes = [input_size] + hidden_layers + [output_size]

        # Initialize weights and biases
        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            if init_method == 'xavier':
                # Xavier/Glorot initialization
                scale = np.sqrt(2.0 / (layer_sizes[i] + layer_sizes[i+1]))
                self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * scale)
            else:
                # Simple random initialization
                self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.01)

            self.biases.append(np.zeros((1, layer_sizes[i+1])))

    def activate(self, h, derivative=False):
        """Apply activation function"""
        if self.activation == 'sigmoid':
            if derivative:
                return self.sigmoid(h) * (1 - self.sigmoid(h))
            return self.sigmoid(h)

        elif self.activation == 'tanh':
            if derivative:
                return 1 - np.power(self.tanh(h), 2)
            return self.tanh(h)

        elif self.activation == 'relu':
            if derivative:
                return np.where(h > 0, 1, 0)
            return np.maximum(0, h)

    def sigmoid(self, h):
        return 1 / (1 + np.exp(-np.clip(h, -500, 500)))  # Clip to avoid overflow

    def tanh(self, h):
        return np.tanh(h)

    def softmax(self, h):
        # Subtract max for numerical stability
        exp_h = np.exp(h - np.max(h, axis=1, keepdims=True))
        return exp_h / np.sum(exp_h, axis=1, keepdims=True)

    def forward(self, X):
        """
        Forward propagation

        Parameters:
        - X: Input data (batch_size x input_size)

        Returns:
        - Activations and pre-activations for backpropagation
        """
        # Store activations and pre-activations for backpropagation
        self.H = []  # Pre-activations
        self.A = [X]  # Activations (A[0] is the input)

        # Forward through hidden layers
        for i in range(len(self.weights) - 1):
            H = np.dot(self.A[i], self.weights[i]) + self.biases[i]
            self.H.append(H)
            self.A.append(self.activate(H))

        # Output layer with softmax
        H_out = np.dot(self.A[-1], self.weights[-1]) + self.biases[-1]
        self.H.append(H_out)
        output = self.softmax(H_out)
        self.A.append(output)

        return output

    def compute_loss(self, y_pred, y_true):
        """
        Compute cross-entropy loss

        Parameters:
        - y_pred: Predicted probabilities (softmax output)
        - y_true: One-hot encoded ground truth

        Returns:
        - Cross-entropy loss
        """
        m = y_true.shape[0]  # Batch size
        # Add small epsilon to avoid log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        cross_entropy = -np.sum(y_true * np.log(y_pred)) / m
        return cross_entropy

    def predict(self, X):
        """
        Make predictions

        Parameters:
        - X: Input data

        Returns:
        - Class predictions
        """
        output = self.forward(X)
        return np.argmax(output, axis=1)

## Question 3

Implement the backpropagation algorithm with support for the following optimisation functions

1. sgd
2. momentum based gradient descent
3. nesterov accelerated gradient descent
4. rmsprop
5. adam
6. nadam

In [5]:
class NeuralNetwork(NeuralNetwork):  # Extending the previous class
    def backpropagation(self, X, y, batch_size=32, learning_rate=0.001,
                        optimizer='sgd', epochs=10, beta1=0.9, beta2=0.999,
                        epsilon=1e-8, weight_decay=0):
        """
        Train the neural network using backpropagation

        Parameters:
        - X: Training data
        - y: Target labels (one-hot encoded)
        - batch_size: Size of mini-batches
        - learning_rate: Learning rate
        - optimizer: 'sgd', 'momentum', 'nag', 'rmsprop', 'adam', or 'nadam'
        - epochs: Number of training epochs
        - beta1: Exponential decay rate for first moment estimates (momentum)
        - beta2: Exponential decay rate for second moment estimates (RMSprop)
        - epsilon: Small constant for numerical stability
        - weight_decay: L2 regularization parameter

        Returns:
        - Training history (loss, accuracy)
        """
        m = X.shape[0]  # Number of training examples
        n_batches = int(np.ceil(m / batch_size))
        history = {'loss': [], 'accuracy': []}

        # Initialize=ing optimizer parameters
        if optimizer in ['momentum', 'nag']:
            # Velocity for momentum and NAG
            velocities_w = [np.zeros_like(w) for w in self.weights]
            velocities_b = [np.zeros_like(b) for b in self.biases]

        if optimizer in ['rmsprop', 'adam', 'nadam']:
            # Cache for RMSprop, Adam, and Nadam
            cache_w = [np.zeros_like(w) for w in self.weights]
            cache_b = [np.zeros_like(b) for b in self.biases]

        if optimizer in ['adam', 'nadam']:
            # Momentum for Adam and Nadam
            moment_w = [np.zeros_like(w) for w in self.weights]
            moment_b = [np.zeros_like(b) for b in self.biases]
            t = 0  # Timestep for bias correction

        for epoch in range(epochs):
            # Shuffle data for each epoch
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            epoch_loss = 0
            correct_preds = 0

            for batch in range(n_batches):
                # Get mini-batch
                start_idx = batch * batch_size
                end_idx = min((batch + 1) * batch_size, m)
                X_batch = X_shuffled[start_idx:end_idx]
                y_batch = y_shuffled[start_idx:end_idx]
                batch_size_actual = X_batch.shape[0]

                # Forward pass
                y_pred = self.forward(X_batch)
                loss = self.compute_loss(y_pred, y_batch)
                epoch_loss += loss * batch_size_actual

                # Calculate accuracy
                y_pred_classes = np.argmax(y_pred, axis=1)
                y_true_classes = np.argmax(y_batch, axis=1)
                correct_preds += np.sum(y_pred_classes == y_true_classes)

                # Backpropagation - compute gradients
                dA = y_pred - y_batch  # Derivative of softmax with cross-entropy

                dW = []
                db = []

                # Output layer gradients
                dW_last = np.dot(self.A[-2].T, dA) / batch_size_actual
                db_last = np.sum(dA, axis=0, keepdims=True) / batch_size_actual

                # Adding L2 regularization
                if weight_decay > 0:
                    dW_last += weight_decay * self.weights[-1]

                dW.insert(0, dW_last)
                db.insert(0, db_last)

                # Backpropagate through hidden layers
                dA_prev = dA
                for i in reversed(range(len(self.weights) - 1)):
                    # Compute dh for the current layer
                    dH = np.dot(dA_prev, self.weights[i+1].T) * self.activate(self.H[i], derivative=True)

                    # Compute gradients
                    dW_curr = np.dot(self.A[i].T, dH) / batch_size_actual
                    db_curr = np.sum(dH, axis=0, keepdims=True) / batch_size_actual

                    # Add L2 regularization
                    if weight_decay > 0:
                        dW_curr += weight_decay * self.weights[i]

                    dW.insert(0, dW_curr)
                    db.insert(0, db_curr)

                    # Update dA for next iteration
                    dA_prev = dH

                # Update weights and biases based on optimizer
                if optimizer == 'sgd':
                    # Standard SGD
                    for i in range(len(self.weights)):
                        self.weights[i] -= learning_rate * dW[i]
                        self.biases[i] -= learning_rate * db[i]

                elif optimizer == 'momentum':
                    # SGD with Momentum
                    for i in range(len(self.weights)):
                        velocities_w[i] = beta1 * velocities_w[i] + (1 - beta1) * dW[i]
                        velocities_b[i] = beta1 * velocities_b[i] + (1 - beta1) * db[i]

                        self.weights[i] -= learning_rate * velocities_w[i]
                        self.biases[i] -= learning_rate * velocities_b[i]

                elif optimizer == 'nag':
                    # Nesterov Accelerated Gradient
                    for i in range(len(self.weights)):
                        # Update velocity first
                        velocities_w[i] = beta1 * velocities_w[i] + (1 - beta1) * dW[i]
                        velocities_b[i] = beta1 * velocities_b[i] + (1 - beta1) * db[i]

                        # "Look ahead" - use the updated velocity for the gradient step
                        self.weights[i] -= learning_rate * (beta1 * velocities_w[i] + (1 - beta1) * dW[i])
                        self.biases[i] -= learning_rate * (beta1 * velocities_b[i] + (1 - beta1) * db[i])

                elif optimizer == 'rmsprop':
                    # RMSprop
                    for i in range(len(self.weights)):
                        # Update cache (moving average of squared gradients)
                        cache_w[i] = beta2 * cache_w[i] + (1 - beta2) * (dW[i] ** 2)
                        cache_b[i] = beta2 * cache_b[i] + (1 - beta2) * (db[i] ** 2)

                        # Update weights and biases
                        self.weights[i] -= learning_rate * dW[i] / (np.sqrt(cache_w[i]) + epsilon)
                        self.biases[i] -= learning_rate * db[i] / (np.sqrt(cache_b[i]) + epsilon)

                elif optimizer == 'adam':
                    # Adam optimizer
                    t += 1  # Increment timestep

                    for i in range(len(self.weights)):
                        # Update first moment estimate (momentum)
                        moment_w[i] = beta1 * moment_w[i] + (1 - beta1) * dW[i]
                        moment_b[i] = beta1 * moment_b[i] + (1 - beta1) * db[i]

                        # Update second moment estimate (RMSprop)
                        cache_w[i] = beta2 * cache_w[i] + (1 - beta2) * (dW[i] ** 2)
                        cache_b[i] = beta2 * cache_b[i] + (1 - beta2) * (db[i] ** 2)

                        # Bias correction
                        moment_w_corrected = moment_w[i] / (1 - beta1 ** t)
                        moment_b_corrected = moment_b[i] / (1 - beta1 ** t)
                        cache_w_corrected = cache_w[i] / (1 - beta2 ** t)
                        cache_b_corrected = cache_b[i] / (1 - beta2 ** t)

                        # Update weights and biases
                        self.weights[i] -= learning_rate * moment_w_corrected / (np.sqrt(cache_w_corrected) + epsilon)
                        self.biases[i] -= learning_rate * moment_b_corrected / (np.sqrt(cache_b_corrected) + epsilon)

                elif optimizer == 'nadam':
                    # Nadam (Adam with Nesterov momentum)
                    t += 1  # Increment timestep

                    for i in range(len(self.weights)):
                        # Update first moment estimate
                        moment_w[i] = beta1 * moment_w[i] + (1 - beta1) * dW[i]
                        moment_b[i] = beta1 * moment_b[i] + (1 - beta1) * db[i]

                        # Update second moment estimate
                        cache_w[i] = beta2 * cache_w[i] + (1 - beta2) * (dW[i] ** 2)
                        cache_b[i] = beta2 * cache_b[i] + (1 - beta2) * (db[i] ** 2)

                        # Bias correction
                        moment_w_corrected = moment_w[i] / (1 - beta1 ** t)
                        moment_b_corrected = moment_b[i] / (1 - beta1 ** t)
                        cache_w_corrected = cache_w[i] / (1 - beta2 ** t)
                        cache_b_corrected = cache_b[i] / (1 - beta2 ** t)

                        # Nesterov momentum update
                        moment_w_nesterov = beta1 * moment_w_corrected + (1 - beta1) * dW[i] / (1 - beta1 ** t)
                        moment_b_nesterov = beta1 * moment_b_corrected + (1 - beta1) * db[i] / (1 - beta1 ** t)

                        # Update weights and biases
                        self.weights[i] -= learning_rate * moment_w_nesterov / (np.sqrt(cache_w_corrected) + epsilon)
                        self.biases[i] -= learning_rate * moment_b_nesterov / (np.sqrt(cache_b_corrected) + epsilon)

            # Calculate epoch statistics
            epoch_loss /= m
            epoch_accuracy = correct_preds / m

            history['loss'].append(epoch_loss)
            history['accuracy'].append(epoch_accuracy)

            print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

        return history

## Question 4:
Use the sweep functionality provided by wandb to find the best values for the hyperparameters listed below. Use the standard train/test split of fashion_mnist (use (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()). Keep 10% of the training data aside as validation data for this hyperparameter search. Here are some suggestions for different values to try for hyperparameters. As you can quickly see that this leads to an exponential number of combinations. You will have to think about strategies to do this hyperparameter search efficiently. Check out the options provided by wandb.sweep and write down what strategy you chose and why.

1. number of epochs: 5, 10
2. number of hidden layers: 3, 4, 5
3. size of every hidden layer: 32, 64, 128
4. weight decay (L2 regularisation): 0, 0.0005, 0.5
5. learning rate: 1e-3, 1 e-4
6. optimizer: sgd, momentum, nesterov, rmsprop, adam, nadam
7. batch size: 16, 32, 64
8. weight initialisation: random, Xavier
9. activation functions: sigmoid, tanh, ReLU


In [7]:
from sklearn.model_selection import train_test_split

def run_wandb_sweep():
    # sweep configuration
    sweep_config = {
        'method': 'bayes',  # Bayesian optimization
        'metric': {
            'name': 'val_accuracy',
            'goal': 'maximize'
        },
        'parameters': {
            'hidden_layers': {
                'values': [3, 4, 5]
            },

            'hidden_layer_size': {
                'values': [32, 64, 128]
            },

            'activation': {
                'values': ['sigmoid', 'tanh', 'relu']
            },
            'optimizer': {
                'values': ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']
            },
            'learning_rate': {
                'distribution': 'log_uniform',
                'min': np.log(1e-4),
                'max': np.log(1e-3)
            },
            'batch_size': {
                'values': [16, 32, 64]
            },
            'epochs': {
                'values': [5, 10]
            },
            'weight_decay': {
                'values': [0, 0.0005, 0.5]
            },
            'init_method': {
                'values': ['random', 'xavier']
            }
        }
    }

    # Initialize sweep
    sweep_id = wandb.sweep(sweep_config, project="DA6401-Assignment1")

    def train_model():
        # Initialize wandb run
        run = wandb.init()

        # Get hyperparameters
        config = wandb.config

        # Load data
        (X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

        # Normalize data
        X_train_full = X_train_full.astype("float32") / 255.0
        X_test = X_test.astype("float32") / 255.0

        # Reshape data (flatten 28x28 to 784)
        X_train_full = X_train_full.reshape(-1, 28*28)
        X_test = X_test.reshape(-1, 28*28)

        # Split into training and validation sets (10% for validation)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full, y_train_full, test_size=0.1, random_state=42
        )

        # One-hot encode labels
        y_train_one_hot = np.eye(10)[y_train]
        y_val_one_hot = np.eye(10)[y_val]

        # Initialize model with the chosen hyperparameters
        model = NeuralNetwork(
            input_size=784,
            hidden_layers=[config.hidden_layer_size] * config.hidden_layers,
            output_size=10,
            activation=config.activation,
            init_method=config.init_method
        )

        # Train model
        history = model.backpropagation(
            X_train, y_train_one_hot,
            batch_size=config.batch_size,
            learning_rate=config.learning_rate,
            optimizer=config.optimizer,
            epochs=config.epochs,
            weight_decay=config.weight_decay
        )

        # Evaluate on validation set
        val_predictions = model.predict(X_val)
        val_accuracy = np.mean(val_predictions == y_val)
        best_val_accuracy=0

        # Log metrics
        for epoch in range(config.epochs):
            val_loss = model.compute_loss(model.forward(X_val), np.eye(10)[y_val])

            # Log metrics to wandb
            wandb.log({
                "epoch": epoch,
                "train_loss": history['loss'][epoch],
                "train_accuracy": history['accuracy'][epoch],
                "val_loss": val_loss,
                # "val_accuracy": history["val_accuracy"][epoch]
            })

        wandb.log({"val_accuracy": val_accuracy})
          # Log the best validation accuracy for this model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            wandb.run.summary["best_val_accuracy"] = best_val_accuracy
            wandb.run.summary["best_model_config"] = dict(config)
            wandb.run.summary["best_model_weights"] = model.weights
            wandb.run.summary["best_model_biases"] = model.biases

        #log best model
        wandb.run.summary["best_val_accuracy"] = best_val_accuracy
        wandb.run.summary["best_model_config"] = dict(config)
        wandb.run.summary["best_model_weights"] = model.weights
        wandb.run.summary["best_model_biases"] = model.biases


        # Create learning curves
        wandb.log({
            "learning_curve": wandb.plot.line_series(
                xs=list(range(1, config.epochs+1)),
                ys=[history['loss'], history['accuracy']],
                keys=["Training Loss", "Training Accuracy"],
                title="Learning Curves",
                xname="Epoch"
            )
        })

        run.finish()

    # Run the sweep
    wandb.agent(sweep_id, train_model, count=50)  # Run 50 trials

if __name__ == "__main__":
    run_wandb_sweep()

wandb: WARNING Malformed sweep config detected! This may cause your sweep to behave in unexpected ways.
wandb: WARNING To avoid this, please fix the sweep config schema violations below:
wandb: WARNING   Violation 1. learning_rate uses log_uniform, where min/max specify base-e exponents. Use log_uniform_values to specify limit values.


Create sweep with ID: 84hhr0lw
Sweep URL: https://wandb.ai/krishvamsi321-indian-institute-of-technology-madras/DA6401-Assignment1/sweeps/84hhr0lw


wandb: Agent Starting Run: b1xau7v0 with config:
wandb: 	activation: tanh
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_layer_size: 64
wandb: 	hidden_layers: 5
wandb: 	init_method: xavier
wandb: 	learning_rate: 0.00011486291865316924
wandb: 	optimizer: nadam
wandb: 	weight_decay: 0.0005


Epoch 1/10, Loss: 0.6065, Accuracy: 0.7939
Epoch 2/10, Loss: 0.4093, Accuracy: 0.8538
Epoch 3/10, Loss: 0.3731, Accuracy: 0.8651
Epoch 4/10, Loss: 0.3524, Accuracy: 0.8738
Epoch 5/10, Loss: 0.3367, Accuracy: 0.8789
Epoch 6/10, Loss: 0.3251, Accuracy: 0.8821
Epoch 7/10, Loss: 0.3154, Accuracy: 0.8847
Epoch 8/10, Loss: 0.3067, Accuracy: 0.8889
Epoch 9/10, Loss: 0.2990, Accuracy: 0.8916
Epoch 10/10, Loss: 0.2939, Accuracy: 0.8931


wandb: WARNING Serializing object of type ndarray that is 401536 bytes
wandb: WARNING Serializing object of type ndarray that is 401536 bytes


...

epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▄▃▃▂▂▂▁▁▁
val_accuracy,▁
val_loss,▁▁▁▁▁▁▁▁▁▁
best_val_accuracy,0.879
epoch,9
train_accuracy,0.90663
train_loss,0.24835
val_accuracy,0.879
val_loss,0.32739


wandb: Agent Starting Run: p8cz5sax with config:
wandb: 	activation: relu
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_layer_size: 128
wandb: 	hidden_layers: 4
wandb: 	init_method: xavier
wandb: 	learning_rate: 0.00016671045095785492
wandb: 	optimizer: nadam
wandb: 	weight_decay: 0.0005


Epoch 1/10, Loss: 0.5443, Accuracy: 0.8104
Epoch 2/10, Loss: 0.3900, Accuracy: 0.8611
Epoch 3/10, Loss: 0.3529, Accuracy: 0.8719
Epoch 4/10, Loss: 0.3296, Accuracy: 0.8802
Epoch 5/10, Loss: 0.3126, Accuracy: 0.8849
Epoch 6/10, Loss: 0.2981, Accuracy: 0.8893
Epoch 7/10, Loss: 0.2874, Accuracy: 0.8940
Epoch 8/10, Loss: 0.2767, Accuracy: 0.8979
Epoch 9/10, Loss: 0.2686, Accuracy: 0.9019
Epoch 10/10, Loss: 0.2597, Accuracy: 0.9036


wandb: WARNING Serializing object of type ndarray that is 802944 bytes
wandb: WARNING Serializing object of type ndarray that is 131200 bytes
wandb: WARNING Serializing object of type ndarray that is 131200 bytes
wandb: WARNING Serializing object of type ndarray that is 131200 bytes
wandb: WARNING Serializing object of type ndarray that is 802944 bytes
wandb: WARNING Serializing object of type ndarray that is 131200 bytes
wandb: WARNING Serializing object of type ndarray that is 131200 bytes
wandb: WARNING Serializing object of type ndarray that is 131200 bytes


epoch,▁▂▃▃▄▅▆▆▇█
train_accuracy,▁▅▆▆▇▇▇███
train_loss,█▄▃▃▂▂▂▁▁▁
val_accuracy,▁
val_loss,▁▁▁▁▁▁▁▁▁▁
best_val_accuracy,0.88433
epoch,9
train_accuracy,0.90361
train_loss,0.25967
val_accuracy,0.88433
val_loss,0.32465


## Question 6
 the best accuracy on the validation set across all the models that you train.

wandb automatically generates this plot which summarises the test accuracy of all the models that you tested.